In [1]:
import pandas as pd
import duckdb

# Get positive patient counts

In [ ]:
cohort_con: duckdb.DuckDBPyConnection = duckdb.connect()

# Get the oncology cohort (all active and past cancer patients)
diagnoses_sql_queries = ["active_cancer", "personal_history_cancer", "history_and_active"]
for query in diagnoses_sql_queries:
    with open(f"exploration_sql_files/diagnoses_sql/{query}.sql", "r") as f:
        sql: str = f.read()
        cohort_con.execute(sql).df()

# Fetch the prescriptions/oncology drugs
with open("exploration_sql_files/prescriptions_sql/prescriptions_count_regex.sql", "r") as f:
    sql: str = f.read()
    cohort_con.execute(sql).df()


# Get the first occurrence of drugs:
with open("exploration_sql_files/cohort_outcomes_sql/first_drug.sql", "r") as f:
    sql: str = f.read()
    cohort_con.execute(sql).df()

# Get the cardiovascular, ICD outcomes:
with open("exploration_sql_files/cohort_outcomes_sql/cardiovascular_outcomes.sql", "r") as f:
    sql: str = f.read()
    cohort_con.execute(sql).df()

# Get the LVEF outcomes:
with open("exploration_sql_files/cohort_outcomes_sql/lvef_outcomes.sql", "r") as f:
    sql: str = f.read()
    cohort_con.execute(sql).df()

# Create the entire cohort:
with open("exploration_sql_files/cohort_outcomes_sql/create_cohorts.sql", "r") as f:
    sql: str = f.read()

    
cohort_con_result: pd.DataFrame = cohort_con.execute(sql).df() # remember to fetch result using df()
cohort_con_result # takes around 1 minute

# Perform labelling on the returned result

In [43]:
def summarize_cardiotoxicity_cohort(cardiotoxicity_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    df = cardiotoxicity_df.copy()

    flag_cols = [
        "cv_event_1yr",
        "has_baseline_lvef",
        "has_followup_lvef_1yr",
        "lvef_drop_10_1yr",
        "lvef_below_50_1yr",
        "definite_lvef_ctrcd_1yr",
        "pre_existing_cv_history",
        "cardiotoxicity_1yr",
        "has_outcome_evidence_1yr",
    ]

    for col in flag_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    date_cols = [
        "first_oncology_time",
        "baseline_time",
        "first_followup_lvef_time_1yr",
    ]

    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    lvef_cols = [
        "baseline_lvef",
        "worst_followup_lvef_1yr",
    ]

    for col in lvef_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    overall_stats = {
        "n_exposed_patients": len(df),
        "n_composite_cardiotoxicity_1yr": df["cardiotoxicity_1yr"].sum(),
        "pct_composite_cardiotoxicity_1yr": df["cardiotoxicity_1yr"].mean() * 100,
        "n_icd_cv_event_1yr": df["cv_event_1yr"].sum(),
        "pct_icd_cv_event_1yr": df["cv_event_1yr"].mean() * 100,
        "n_lvef_ctrcd_1yr": df["definite_lvef_ctrcd_1yr"].sum(),
        "pct_lvef_ctrcd_1yr": df["definite_lvef_ctrcd_1yr"].mean() * 100,
        "n_lvef_drop_10_1yr": df["lvef_drop_10_1yr"].sum(),
        "pct_lvef_drop_10_1yr": df["lvef_drop_10_1yr"].mean() * 100,
        "n_lvef_below_50_1yr": df["lvef_below_50_1yr"].sum(),
        "pct_lvef_below_50_1yr": df["lvef_below_50_1yr"].mean() * 100,
        "n_with_baseline_lvef": df["has_baseline_lvef"].sum(),
        "pct_with_baseline_lvef": df["has_baseline_lvef"].mean() * 100,
        "n_with_followup_lvef_1yr": df["has_followup_lvef_1yr"].sum(),
        "pct_with_followup_lvef_1yr": df["has_followup_lvef_1yr"].mean() * 100,
        "n_with_outcome_evidence_1yr": df["has_outcome_evidence_1yr"].sum(),
        "pct_with_outcome_evidence_1yr": df["has_outcome_evidence_1yr"].mean() * 100,
        "n_pre_existing_cv_history": df["pre_existing_cv_history"].sum(),
        "pct_pre_existing_cv_history": df["pre_existing_cv_history"].mean() * 100,
    }

    overall_stats_df = pd.DataFrame(
        overall_stats.items(),
        columns=["metric", "value"]
    )

    model_df = df[
        (df["has_outcome_evidence_1yr"] == 1)
        & (df["pre_existing_cv_history"] == 0)
    ].copy()

    model_stats = {
        "n_model_cohort": len(model_df),
        "n_cardiotoxicity_cases": model_df["cardiotoxicity_1yr"].sum(),
        "pct_cardiotoxicity_cases": model_df["cardiotoxicity_1yr"].mean() * 100,
        "n_controls": len(model_df) - model_df["cardiotoxicity_1yr"].sum(),
        "pct_controls": (1 - model_df["cardiotoxicity_1yr"].mean()) * 100,
        "n_icd_cv_event_1yr": model_df["cv_event_1yr"].sum(),
        "n_lvef_ctrcd_1yr": model_df["definite_lvef_ctrcd_1yr"].sum(),
        "n_with_baseline_lvef": model_df["has_baseline_lvef"].sum(),
        "n_with_followup_lvef_1yr": model_df["has_followup_lvef_1yr"].sum(),
    }

    model_stats_df = pd.DataFrame(
        model_stats.items(),
        columns=["metric", "value"]
    )

    case_control_summary = (
        model_df
        .groupby("cardiotoxicity_1yr")
        .agg(
            n_patients=("subject_id", "nunique"),
            n_cv_event_1yr=("cv_event_1yr", "sum"),
            n_lvef_ctrcd_1yr=("definite_lvef_ctrcd_1yr", "sum"),
            n_lvef_drop_10_1yr=("lvef_drop_10_1yr", "sum"),
            n_lvef_below_50_1yr=("lvef_below_50_1yr", "sum"),
            n_with_baseline_lvef=("has_baseline_lvef", "sum"),
            n_with_followup_lvef_1yr=("has_followup_lvef_1yr", "sum"),
            mean_baseline_lvef=("baseline_lvef", "mean"),
            median_baseline_lvef=("baseline_lvef", "median"),
            mean_worst_followup_lvef=("worst_followup_lvef_1yr", "mean"),
            median_worst_followup_lvef=("worst_followup_lvef_1yr", "median"),
        )
        .reset_index()
    )

    lvef_patients = df[
        (df["has_baseline_lvef"] == 1)
        & (df["has_followup_lvef_1yr"] == 1)
    ].copy()

    lvef_patients["absolute_lvef_drop"] = (
        lvef_patients["baseline_lvef"]
        - lvef_patients["worst_followup_lvef_1yr"]
    )

    lvef_summary = lvef_patients[
        [
            "baseline_lvef",
            "worst_followup_lvef_1yr",
            "absolute_lvef_drop",
        ]
    ].describe()

    df["days_to_first_followup_lvef"] = (
        df["first_followup_lvef_time_1yr"]
        - df["first_oncology_time"]
    ).dt.days

    followup_time_summary = (
        df.loc[
            df["has_followup_lvef_1yr"] == 1,
            "days_to_first_followup_lvef"
        ]
        .describe()
        .to_frame(name="days_to_first_followup_lvef")
    )

    return {
        "overall_stats": overall_stats_df,
        "model_stats": model_stats_df,
        "case_control_summary": case_control_summary,
        "lvef_summary": lvef_summary,
        "followup_time_summary": followup_time_summary,
    }

In [44]:
cardiotoxicity_df = cohort_con_result.copy()

In [45]:
summaries = summarize_cardiotoxicity_cohort(cardiotoxicity_df)

summaries["overall_stats"]

,metric,value
0,n_exposed_patients,2545.000000
1,n_composite_cardiotoxicity_1yr,399.000000
2,pct_composite_cardiotoxicity_1yr,15.677800
3,n_icd_cv_event_1yr,367.000000
4,pct_icd_cv_event_1yr,14.420432
5,n_lvef_ctrcd_1yr,95.000000
6,pct_lvef_ctrcd_1yr,3.732809
7,n_lvef_drop_10_1yr,256.000000
8,pct_lvef_drop_10_1yr,10.058939
9,n_lvef_below_50_1yr,177.000000


In [46]:
summaries["model_stats"]

,metric,value
0,n_model_cohort,965.000000
1,n_cardiotoxicity_cases,184.000000
2,pct_cardiotoxicity_cases,19.067358
3,n_controls,781.000000
4,pct_controls,80.932642
5,n_icd_cv_event_1yr,162.000000
6,n_lvef_ctrcd_1yr,57.000000
7,n_with_baseline_lvef,556.000000
8,n_with_followup_lvef_1yr,938.000000


In [47]:
summaries["case_control_summary"]

,cardiotoxicity_1yr,n_patients,n_cv_event_1yr,n_lvef_ctrcd_1yr,n_lvef_drop_10_1yr,n_lvef_below_50_1yr,n_with_baseline_lvef,n_with_followup_lvef_1yr,mean_baseline_lvef,median_baseline_lvef,mean_worst_followup_lvef,median_worst_followup_lvef
0,0,781,0,0,117,20,440,781,61.938636,60.0,58.882202,59.0
1,1,184,162,57,75,75,116,157,60.818966,60.0,47.401274,50.0


In [48]:
summaries["lvef_summary"]

,baseline_lvef,worst_followup_lvef_1yr,absolute_lvef_drop
count,706.000000,706.000000,706.000000
mean,60.381020,54.386686,5.994334
std,8.313239,10.573447,10.523191
min,20.000000,15.000000,-25.000000
25%,55.000000,50.000000,0.000000
50%,60.000000,55.000000,5.000000
75%,65.000000,60.000000,10.000000
max,80.000000,80.000000,50.000000


In [49]:
summaries["followup_time_summary"]

,days_to_first_followup_lvef
count,1169.000000
mean,62.372968
std,72.326039
min,0.000000
25%,12.000000
50%,36.000000
75%,85.000000
max,345.000000
